# Lab 3. Hyperparameter optimization

Tune a model from lab 2 with cross-validation and compare Grid Search with Random Search.

- **Task:** Titanic survival (lab 2), the same features, preprocessing and train/test split (`random_state=42`).
- **Model:** `GradientBoostingClassifier`. It has several hyperparameters with a clear meaning and was one of the best models in lab 2.
- **Metric:** F1 for `Survived = 1`.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import loguniform, randint
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    RepeatedStratifiedKFold,
    cross_val_score,
    train_test_split,
    validation_curve,
)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.notebook_repr_html", False)
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 72

SEED = 42
TARGET = "Survived"

## 1. Setup and baseline

The **baseline** of this lab is gradient boosting with default parameters (`n_estimators=100`, `learning_rate=0.1`, `max_depth=3`). Its CV F1 and test F1 are fixed before any tuning.

Its CV F1 here (0.757) differs from lab 2 (0.747) only because the CV scheme differs: 15 folds instead of 5 (section 3). The test F1 (0.738) is the same.

In [ ]:
titanic = pd.read_csv("../lab2/data/titanic.csv")
titanic["Title"] = (
    titanic["Name"]
    .str.extract(r",\s*([^.]+)\.", expand=False)
    .replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
)
titanic.loc[~titanic["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Title"] = "Rare"
titanic["FamilySize"] = titanic["SibSp"] + titanic["Parch"] + 1

NUMERIC = ["Pclass", "Age", "Fare", "FamilySize"]
CATEGORICAL = ["Sex", "Embarked", "Title"]
train, test = train_test_split(titanic, test_size=0.2, stratify=titanic[TARGET], random_state=SEED)
X_train, y_train = train[NUMERIC + CATEGORICAL], train[TARGET]
X_test, y_test = test[NUMERIC + CATEGORICAL], test[TARGET]

preprocessor = ColumnTransformer(
    [
        ("num", SimpleImputer(strategy="median", add_indicator=True), NUMERIC),
        (
            "cat",
            make_pipeline(
                SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")
            ),
            CATEGORICAL,
        ),
    ]
)
baseline = Pipeline(
    [("prep", preprocessor), ("model", GradientBoostingClassifier(random_state=SEED))]
)
CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)


def short(params: dict) -> str:
    return ", ".join(
        f"{k.removeprefix('model__')}={round(v, 4) if isinstance(v, float) else v}"
        for k, v in sorted(params.items())
    )


baseline_folds = cross_val_score(baseline, X_train, y_train, cv=CV, scoring="f1")
baseline_test = f1_score(y_test, clone(baseline).fit(X_train, y_train).predict(X_test))
print(f"train: {len(train)}, test: {len(test)}")
print(f"baseline (default GB): CV F1 {baseline_folds.mean():.3f} +- {baseline_folds.std():.3f}")

## 2. Hyperparameters

| Hyperparameter | Meaning | Range | Expected effect |
| --- | --- | --- | --- |
| `n_estimators` | number of trees, i.e. boosting steps | 50–400 | more trees → lower error on train; too many → the model starts fitting noise |
| `learning_rate` | how much each new tree corrects the previous ones | 0.01–0.3, log scale | small → careful learning that needs many trees; large → fast, but overshoots and overfits |
| `max_depth` | depth of each tree = how many features one tree can combine | 1–6 | 1 = no interactions (sex and class separately); 3–4 = interactions like "3rd class woman"; deeper → memorizing individual passengers |

`n_estimators` and `learning_rate` work together: the total amount of learning is roughly their product, so a small step needs more trees. Both searches use exactly these ranges.

## 3. Cross-validation

**Scheme:** `RepeatedStratifiedKFold(n_splits=5, n_repeats=3)` — 15 estimates for every parameter set, computed on the **training part only**. The test set does not take part in the search.

- **Why 5 folds:** each model is trained on 570 passengers (80% of train, close to the final size) and validated on 142 (about 55 survivors). With 10 folds a validation fold would have only ~27 survivors, and F1 on it would be very noisy.
- **Why 3 repeats:** the data is small, and the result depends on which passengers land in which fold. Three different splits average this out. On 712 rows it is cheap.
- All models are evaluated on **the same 15 folds**, so they can be compared fold by fold (section 7).

**Answers**

- **Why cross-validation is needed:** to estimate quality on data the model did not see, using every training object for validation once and averaging several estimates instead of trusting one.
- **Why one train/test split is not enough:** with 712 passengers, F1 on one validation part varies by ±0.04 depending on the split (see the baseline std above). That is larger than the difference between good parameter sets, so on one split we would simply pick the luckiest one.
- **What happens if we tune on test many times:** we fit the parameters to the 179 test passengers, the test score becomes optimistic, and there is no honest estimate of quality on new data left.

## 4. Grid Search

In [ ]:
grid_space = {
    "model__n_estimators": [50, 100, 200, 400],
    "model__learning_rate": [0.01, 0.03, 0.1, 0.3],
    "model__max_depth": [1, 2, 3, 4, 5, 6],
}
grid = GridSearchCV(baseline, grid_space, cv=CV, scoring="f1", n_jobs=-1, return_train_score=True)
start = time.perf_counter()
grid.fit(X_train, y_train)
grid_time = time.perf_counter() - start

print(
    f"combinations: {len(grid.cv_results_['params'])}, fits: {len(grid.cv_results_['params']) * 15}"
)
print(f"time: {grid_time:.1f} s")
print(f"best CV F1: {grid.best_score_:.3f}")
print(f"best params: {short(grid.best_params_)}")

## 5. Random Search

In [ ]:
random_space = {
    "model__n_estimators": randint(50, 401),
    "model__learning_rate": loguniform(0.01, 0.3),
    "model__max_depth": randint(1, 7),
}
search = RandomizedSearchCV(
    baseline,
    random_space,
    n_iter=len(grid.cv_results_["params"]),
    cv=CV,
    scoring="f1",
    n_jobs=-1,
    random_state=SEED,
    return_train_score=True,
)
start = time.perf_counter()
search.fit(X_train, y_train)
random_time = time.perf_counter() - start

pd.DataFrame(
    {
        "combinations": [len(grid.cv_results_["params"]), len(search.cv_results_["params"])],
        "time, s": [grid_time, random_time],
        "best CV F1": [grid.best_score_, search.best_score_],
        "best params": [short(grid.best_params_), short(search.best_params_)],
    },
    index=["Grid Search", "Random Search"],
)

**Findings**

- **Grid Search** tried all 4 × 4 × 6 = **96 combinations** (1440 fits). Best CV F1 **0.773** with `learning_rate=0.01, max_depth=4, n_estimators=400`.
- **Random Search** got the same budget: 96 random combinations from the same ranges. Best CV F1 **0.774** with `learning_rate≈0.047, max_depth=4, n_estimators=93` — a learning rate that the grid does not contain.
- **Quality:** practically equal (difference 0.001, far below the fold spread of ~0.04).
- **Time:** comparable (about 42 s vs 51 s). The time depends on the size of the sampled models (number and depth of trees), not only on the number of combinations.
- **Flat optimum:** very different settings reach the same CV — 400 trees with lr 0.01, 93 trees with lr 0.047, or 400 trees with lr 0.1 and depth 2 (0.770, section 8). There is no single "right" point, only a wide good region.

**When Random Search is better than a full grid:**

- **Many hyperparameters, few of them important.** A grid wastes most combinations repeating the same values of the important ones, while every random combination tries a new value of each parameter.
- **Continuous parameters** such as `learning_rate`: a grid checks only the listed points, a random search any value in the range. This is exactly how it found lr ≈ 0.047 here.
- **Limited time:** the number of iterations is set in advance and the search can be stopped at any moment, while a grid grows multiplicatively with every new parameter.

## 6. Influence of hyperparameters

In [ ]:
curves = {
    "max_depth": (np.arange(1, 11), False),
    "n_estimators": (np.array([10, 25, 50, 100, 200, 400, 800, 1600]), True),
    "learning_rate": (np.logspace(-3, 0, 10), True),
}
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (param, (values, log_scale)) in zip(axes, curves.items(), strict=True):
    train_scores, cv_scores = validation_curve(
        baseline,
        X_train,
        y_train,
        param_name=f"model__{param}",
        param_range=values,
        cv=CV,
        scoring="f1",
        n_jobs=-1,
    )
    for scores, label, color in [(train_scores, "train", "#d6604d"), (cv_scores, "CV", "#2166ac")]:
        mean, std = scores.mean(axis=1), scores.std(axis=1)
        ax.plot(values, mean, "o-", color=color, label=label)
        ax.fill_between(values, mean - std, mean + std, color=color, alpha=0.15)
    if log_scale:
        ax.set_xscale("log")
    ax.set_xlabel(param)
    ax.set_ylabel("F1")
    ax.set_title(f"{param} (others default)")
    ax.legend()
fig.tight_layout()
plt.show()

Each curve changes one hyperparameter while the others stay at their defaults (`n_estimators=100`, `learning_rate=0.1`, `max_depth=3`). The bands are ± one standard deviation over the 15 folds.

| Hyperparameter | Underfitting | Reasonable complexity | Overfitting |
| --- | --- | --- | --- |
| `max_depth` | not reached: even depth 1 gives CV 0.76, because sex and class alone already separate most passengers | 1–4 (best ≈ 0.76 at 4) | ≥ 5: train 0.97–0.98, CV slowly falls to 0.73 at depth 10 |
| `n_estimators` | 10: train and CV both low (0.77 / 0.73) | 50–200: CV ≈ 0.76 | ≥ 400: train → 0.98 while CV slightly decreases (within its spread) |
| `learning_rate` | ≤ 0.002: **F1 < 0.1**, the model almost never predicts a survivor | 0.01–0.2 | ≈ 0.5–1: train → 0.97, CV drops to about 0.73 |

**How to read the curves:**

- **Underfitting:** train and CV are both low. The model is too simple even for the training data — for example, with a tiny learning rate 100 trees cannot move the prediction away from "most passengers died".
- **Reasonable complexity:** CV is at its maximum, and the gap between train and CV is moderate.
- **Overfitting:** train keeps growing towards 1.0 while CV goes down. The model memorizes individual passengers, which does not transfer to new ones.

Small wiggles of the CV curve (e.g. depth 2 slightly below depth 1) are within the ±1 std band and are not real effects. The CV curve is almost flat over a wide range (0.74–0.77): on 712 passengers gradient boosting is not very sensitive to tuning as long as the extremes are avoided.

## 7. Final models

In [ ]:
def fold_scores(searcher: GridSearchCV | RandomizedSearchCV) -> np.ndarray:
    results, best = searcher.cv_results_, searcher.best_index_
    return np.array([results[f"split{i}_test_score"][best] for i in range(CV.get_n_splits())])


folds = {
    "Baseline (default)": baseline_folds,
    "Grid Search": fold_scores(grid),
    "Random Search": fold_scores(search),
}
test_f1 = {
    "Baseline (default)": baseline_test,
    "Grid Search": f1_score(y_test, grid.predict(X_test)),
    "Random Search": f1_score(y_test, search.predict(X_test)),
}
final = pd.DataFrame(
    {
        "CV F1": {name: scores.mean() for name, scores in folds.items()},
        "CV std": {name: scores.std() for name, scores in folds.items()},
        "CV min": {name: scores.min() for name, scores in folds.items()},
        "test F1": test_f1,
    }
)
final

In [ ]:
paired = pd.DataFrame(
    {
        name: {
            "mean gain vs baseline": (folds[name] - baseline_folds).mean(),
            "std of gain": (folds[name] - baseline_folds).std(),
            "folds improved": f"{(folds[name] > baseline_folds).sum()} of {len(baseline_folds)}",
        }
        for name in ["Grid Search", "Random Search"]
    }
).T
paired

**Findings**

- **Fold-by-fold comparison.** The CV standard deviations overlap, but that is not the right test for models evaluated on the same folds. The second table compares them fold by fold: on average tuning adds **+0.016**, and the tuned models beat the baseline on **10–11 of the 15 folds**. The gain is small, and it is not strong evidence either: the 15 folds reuse the same 712 passengers three times, and the winner of 96 configurations was chosen by this very CV, so its gain is optimistic (see below).
- **On test it did not help:** 0.738 → 0.734 (Grid) and 0.719 (Random).
- **Why the test disagrees:**
  1. *The test is small.* 179 passengers with 69 survivors; one passenger changes F1 by about 0.01. The drop of 0.004 is less than one passenger, and 0.019 is two. Both are within noise.
  2. *Winner's curse.* The best of 96 CV scores is partly lucky: the configuration that won also happened to fit these folds well, so its CV score is slightly optimistic.
  3. *The ceiling of the features.* The flat validation curves suggest that default boosting is already close to what sex, class, age and title can tell about survival.
- **This is the honest outcome.** Choosing the parameters by test would give a nicer number, but it would be a fit to the test, not a better model.

## 8. Stability

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
sns.stripplot(
    data=pd.DataFrame(folds).melt(var_name="model", value_name="F1"),
    x="F1",
    y="model",
    size=7,
    alpha=0.7,
    ax=ax,
)
ax.set_title("F1 on each of the 15 CV folds")
plt.show()

top = pd.DataFrame(grid.cv_results_).sort_values("mean_test_score", ascending=False).head(6)
top["params"] = top["params"].map(short)
top[["params", "mean_test_score", "std_test_score", "mean_train_score"]].set_index("params")

**Findings**

- **The split matters more than the parameters.** The same model gets very different F1 on different folds — the baseline from 0.69 to 0.83, the Grid Search model from 0.72 to 0.87 (std ≈ 0.04) — while the means of the three models differ by only 0.017.
- **Same mean, different stability.** The six best grid configurations have practically equal mean F1 (0.767–0.773), but their std ranges from 0.032 to 0.043.
  - `learning_rate=0.1, max_depth=2, 400 trees` has the smallest spread (std 0.032);
  - the Grid Search winner (depth 4) has std 0.043.
  
  With 15 overlapping folds such a difference is weak evidence, but it shows what to look at.
  
  Two models with equal mean can differ in how much they depend on the particular data: one scores evenly, the other high on some folds and noticeably lower on others. On new data the second one is less predictable.
- **The selection rule** was fixed in advance: the highest mean CV score (the default of both searches). With differences this small, a reasonable alternative is to prefer the most stable configuration among those close to the best. That would pick the depth-2 model at the cost of 0.003 in mean CV.

## Conclusions

| Model | Parameters | CV F1 | CV std | Test F1 |
| --- | --- | --- | --- | --- |
| Baseline | 100 trees, lr 0.1, depth 3 | 0.757 | 0.039 | **0.738** |
| Grid Search | 400 trees, lr 0.01, depth 4 | 0.773 | 0.043 | 0.734 |
| Random Search | 93 trees, lr 0.047, depth 4 | **0.774** | 0.045 | 0.719 |

- Hyperparameters were chosen only by repeated 5-fold CV on train; the test was used once for each final model.
- Grid Search and Random Search found equally good solutions with the same budget and comparable time. Random Search found a learning rate the grid could not contain; it is preferable when there are many parameters or continuous ranges.
- Validation curves show the classic picture: underfitting with too few trees or a too small learning rate, overfitting with too deep trees or a too large learning rate (and a slight decline with too many trees), and a wide plateau in between.
- Tuning gave a small CV gain (+0.016, better on 10–11 folds of 15). On the 179-passenger test the difference is within noise, and the flat curves suggest that default boosting is already close to the limit of the features.